# Linear and Regularized Regression on the CarDekho Used-Car Dataset

## Notebook 01: Data Understanding and Provenance Check

**Author:** Muhammad Waqas  
**Dataset:** CarDekho Used-Car Dataset  
**Local File:** `data/raw/car_price_data.csv`  
**Kaggle Source:** https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho  

## Objective

The objective of this notebook is to understand the dataset structure,
features, target variable, missing values, and data types before performing
exploratory analysis or regression modeling.

The dataset is described on Kaggle as used-car data collected from websites.
Its source information, logical consistency, and suitability for research
will be evaluated before it is accepted for final modeling.

The `Price` column is expected to be used as the regression target.
This will be confirmed after inspecting the actual dataset columns.

## 1. Import Required Library and Load the Dataset

Pandas is used to load, organize, and inspect tabular data in the form
of a DataFrame.

In [1]:
import pandas as pd

file_path = "../data/raw/car_price_data.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")

Dataset loaded successfully.


## 2. Dataset Dimensions

The dataset dimensions show the total number of observations and variables.

- Rows represent individual car records.
- Columns represent available variables.

In [2]:
rows, columns = df.shape

print("Number of rows:", rows)
print("Number of columns:", columns)

Number of rows: 2059
Number of columns: 20


## 3. Column Names

Column names are inspected to identify the target variable, input features,
identifiers, and potentially unnecessary variables.

In [3]:
print(df.columns.tolist())

['Make', 'Model', 'Price', 'Year', 'Kilometer', 'Fuel Type', 'Transmission', 'Location', 'Color', 'Owner', 'Seller Type', 'Engine', 'Max Power', 'Max Torque', 'Drivetrain', 'Length', 'Width', 'Height', 'Seating Capacity', 'Fuel Tank Capacity']


## 4. Preview of the Dataset

The first five records are inspected to understand the values, formats,
units, and possible data-cleaning requirements.

In [4]:
df.head()

,Make,Model,Price,Year,Kilometer,Fuel Type,Transmission,Location,Color,Owner,Seller Type,Engine,Max Power,Max Torque,Drivetrain,Length,Width,Height,Seating Capacity,Fuel Tank Capacity
0,Honda,Amaze 1.2 VX i-VTEC,505000,2017,87150,Petrol,Manual,Pune,Grey,First,Corporate,1198 cc,87 bhp @ 6000 rpm,109 Nm @ 4500 rpm,FWD,3990.0,1680.0,1505.0,5.0,35.0
1,Maruti Suzuki,Swift DZire VDI,450000,2014,75000,Diesel,Manual,Ludhiana,White,Second,Individual,1248 cc,74 bhp @ 4000 rpm,190 Nm @ 2000 rpm,FWD,3995.0,1695.0,1555.0,5.0,42.0
2,Hyundai,i10 Magna 1.2 Kappa2,220000,2011,67000,Petrol,Manual,Lucknow,Maroon,First,Individual,1197 cc,79 bhp @ 6000 rpm,112.7619 Nm @ 4000 rpm,FWD,3585.0,1595.0,1550.0,5.0,35.0
3,Toyota,Glanza G,799000,2019,37500,Petrol,Manual,Mangalore,Red,First,Individual,1197 cc,82 bhp @ 6000 rpm,113 Nm @ 4200 rpm,FWD,3995.0,1745.0,1510.0,5.0,37.0
4,Toyota,Innova 2.4 VX 7 STR [2016-2020],1950000,2018,69000,Diesel,Manual,Mumbai,Grey,First,Individual,2393 cc,148 bhp @ 3400 rpm,343 Nm @ 1400 rpm,RWD,4735.0,1830.0,1795.0,7.0,55.0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2059 entries, 0 to 2058
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Make                2059 non-null   object 
 1   Model               2059 non-null   object 
 2   Price               2059 non-null   int64  
 3   Year                2059 non-null   int64  
 4   Kilometer           2059 non-null   int64  
 5   Fuel Type           2059 non-null   object 
 6   Transmission        2059 non-null   object 
 7   Location            2059 non-null   object 
 8   Color               2059 non-null   object 
 9   Owner               2059 non-null   object 
 10  Seller Type         2059 non-null   object 
 11  Engine              1979 non-null   object 
 12  Max Power           1979 non-null   object 
 13  Max Torque          1979 non-null   object 
 14  Drivetrain          1923 non-null   object 
 15  Length              1995 non-null   float64
 16  Width 

### Initial Structural Findings

- The dataset contains 2,059 observations and 20 columns.
- `Price`, `Year`, and `Kilometer` are stored as integer variables.
- Five vehicle-dimension and capacity features are stored as decimal variables.
- Twelve columns are stored as object variables.
- `Engine`, `Max Power`, and `Max Torque` contain numerical information
  combined with measurement units and will require feature extraction.
- Missing values are present in technical specification columns.
- Missing-value treatment will be performed later through a preprocessing
  pipeline to prevent data leakage.

## 6. Missing-Value Summary

Missing values are counted and converted into percentages to measure
the severity of incomplete data in each variable.

In [6]:
missing_count = df.isnull().sum()

missing_percent = (missing_count / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percent": missing_percent
})

missing_summary[missing_summary["Missing Count"] > 0]

,Missing Count,Missing Percent
Engine,80,3.89
Max Power,80,3.89
Max Torque,80,3.89
Drivetrain,136,6.61
Length,64,3.11
Width,64,3.11
Height,64,3.11
Seating Capacity,64,3.11
Fuel Tank Capacity,113,5.49


### Missing-Value Observation

- Missing percentages range from 3.11% to 6.61%.
- `Drivetrain` has the highest missing percentage at 6.61%.
- `Engine`, `Max Power`, and `Max Torque` have the same 80 missing records,
  suggesting that their missingness may be related.
- Vehicle dimensions also share the same number of missing records.
- The missing percentages are manageable, so rows will not be removed blindly.
- Numerical and categorical imputation strategies will be evaluated during
  preprocessing.

## 7. Duplicate Record Check

Exact duplicate rows are checked because duplicated observations can bias
the model and produce misleading evaluation results.

In [7]:
duplicate_count = df.duplicated().sum()

print("Number of exact duplicate rows:", duplicate_count)

Number of exact duplicate rows: 0


### Duplicate Observation

No exact duplicate records were found. Therefore, no rows need to be
removed at this stage.

## 8. Target and Candidate Input Features

`Price` is the continuous regression target. All remaining columns are
initially treated as candidate input features. Their final suitability
will be decided after data-quality analysis.

In [8]:
target_column = "Price"

X = df.drop(columns=["Price"])
y = df["Price"]

print("Target variable:", target_column)
print("Number of candidate features:", X.shape[1])
print("X shape:", X.shape)
print("y shape:", y.shape)

Target variable: Price
Number of candidate features: 19
X shape: (2059, 19)
y shape: (2059,)


## 9. Raw Numerical and Categorical Features

Features are initially grouped according to their current Pandas data types.

This is a preliminary grouping because some object columns, such as Engine,
Max Power, and Max Torque, contain numerical information combined with text
and measurement units.

In [9]:
numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Raw numerical columns:")
print(numerical_columns)

print("\nRaw categorical/object columns:")
print(categorical_columns)

Raw numerical columns:
['Year', 'Kilometer', 'Length', 'Width', 'Height', 'Seating Capacity', 'Fuel Tank Capacity']

Raw categorical/object columns:
['Make', 'Model', 'Fuel Type', 'Transmission', 'Location', 'Color', 'Owner', 'Seller Type', 'Engine', 'Max Power', 'Max Torque', 'Drivetrain']


### Feature-Type Observation

- Seven candidate features are currently stored as numerical variables.
- Twelve candidate features are stored as object variables.
- Engine, power, and torque specifications require numerical feature
  extraction before modeling.
- The remaining object variables will later require categorical encoding.
- The final number of model features will increase after feature extraction
  and categorical encoding.

## 10. Descriptive Statistics of Numerical Variables

Descriptive statistics are used to examine central tendency, variation,
range, quartiles, and potentially unrealistic values in numerical variables.

In [10]:
df.describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
Price,2059.0,1702991.70,2419880.64,49000.0,484999.00,825000.0,1925000.0,35000000.0
Year,2059.0,2016.43,3.36,1988.0,2014.00,2017.0,2019.0,2022.0
Kilometer,2059.0,54224.71,57361.72,0.0,29000.00,50000.0,72000.0,2000000.0
Length,1995.0,4280.86,442.46,3099.0,3985.00,4370.0,4629.0,5569.0
Width,1995.0,1767.99,135.27,1475.0,1695.00,1770.0,1831.5,2220.0
Height,1995.0,1591.74,136.07,1165.0,1485.00,1545.0,1675.0,1995.0
Seating Capacity,1995.0,5.31,0.82,2.0,5.00,5.0,5.0,8.0
Fuel Tank Capacity,1946.0,52.00,15.11,15.0,41.25,50.0,60.0,105.0


### Numerical Summary Observation

- Price ranges from 49,000 to 35,000,000.
- The mean Price is considerably higher than the median Price, indicating
  a right-skewed target distribution and the possible presence of luxury cars.
- Vehicle years range from 1988 to 2022, while most vehicles were manufactured
  between 2014 and 2019.
- Kilometer has a maximum value of 2,000,000, which requires investigation
  as a possible extreme value or data-entry issue.
- Vehicle dimensions, seating capacity, and fuel-tank capacity appear
  reasonably plausible based on their observed ranges.
- Missing counts in the technical specification columns are consistent
  with the earlier missing-value analysis.

## 11. Investigation of Extreme Price Values

The most expensive vehicle records are inspected to determine whether
the maximum prices represent valid luxury vehicles or data-entry errors.

In [12]:
price_columns = [
    "Make",
    "Model",
    "Price",
    "Year",
    "Kilometer"
]

df.sort_values(
    by="Price",
    ascending=False
)[price_columns].head(10)

,Make,Model,Price,Year,Kilometer
483,Ferrari,488 GTB,35000000,2018,9500
1305,Land Rover,Range Rover 3.0 V6 Diesel Vogue LWB,27500000,2020,11000
510,Lamborghini,Huracan LP 610-4,24000000,2016,6000
582,Land Rover,Range Rover 3.0 V6 Diesel Vogue,22000000,2019,35000
1369,Rolls-Royce,Ghost Extended Wheelbase,20000000,2012,55000
1246,Rolls-Royce,Ghost Extended Wheelbase,20000000,2011,27000
1890,Mercedes-Benz,S-Class S 450,20000000,2021,6600
1313,Land Rover,Range Rover 3.0 V6 Diesel Vogue,19300000,2019,63000
1912,Mercedes-Benz,S-Class S 350D [2018-2020],18500000,2021,5000
442,Mercedes-Benz,S-Class Maybach S 560,18500000,2021,21000


### Extreme Price Finding

The highest prices belong to recognized luxury and high-performance vehicles,
including Ferrari, Lamborghini, Rolls-Royce, Range Rover, and Mercedes-Benz.

Therefore, these extreme prices appear contextually valid and will not be
removed merely because they are statistically large.

However, the strong price skewness may affect linear regression. A logarithmic
target transformation will be evaluated later and compared with the original
Price scale.

## 12. Investigation of Extreme Kilometer Values

Vehicles with the highest recorded mileage are inspected to determine
whether the values are plausible or likely data-entry errors.

In [13]:
kilometer_columns = [
    "Make",
    "Model",
    "Price",
    "Year",
    "Kilometer",
    "Owner"
]

df.sort_values(
    by="Kilometer",
    ascending=False
)[kilometer_columns].head(10)

,Make,Model,Price,Year,Kilometer,Owner
1125,Renault,Duster 110 PS RXZ 4X2 MT Diesel,450000,2016,2000000,First
1866,Hyundai,Verna 1.6 CRDI SX,925000,2018,925000,First
1515,Hyundai,Creta SX 1.6 AT CRDi,1240000,2018,440000,First
1992,Toyota,Innova 2.0 G1 BS-IV,555000,2010,261236,First
733,Audi,A6 3.0 TDI quattro Premium,1650000,2012,240000,First
1628,Toyota,Innova 2.5 V 7 STR,960000,2012,222000,First
1991,Toyota,Fortuner 2.8 4x4 AT [2016-2020],2750000,2017,219000,Second
181,Toyota,Fortuner 3.0 4x4 MT,1675000,2014,211000,First
134,BMW,5-Series 520d Sedan,1490000,2013,195000,First
388,Tata,Grande GX,199000,2011,192326,Third


### Extreme Kilometer Finding

The records containing 2,000,000 km and 925,000 km appear highly unusual
for their vehicle years and ownership status.

The Hyundai Verna record has the same numerical value for Price and Kilometer,
which may indicate a data-entry or copying error. However, the original values
will not be manually corrected without supporting evidence.

These records will be flagged for later sensitivity analysis. Model performance
will be compared before and after applying a documented outlier-treatment
strategy.

## 13. Categorical Feature Cardinality

Cardinality represents the number of unique values in a categorical feature.
High-cardinality variables require careful encoding because they can create
a large number of model features.

unique_counts = df[categorical_columns].nunique()

unique_counts.sort_values()

### Cardinality Observation

- Transmission, Seller Type, Drivetrain, and Owner are low-cardinality
  categorical features.
- Make and Color have manageable cardinality.
- Location has moderately high cardinality.
- Model has 1,050 unique values among 2,059 observations and may cause
  overfitting if directly one-hot encoded.
- Engine, Max Power, and Max Torque will not be treated as categorical
  features because they contain numerical technical specifications.
- Alternative strategies for Model will be compared, including exclusion,
  rare-category grouping, or controlled encoding.

## 14. Values of Low-Cardinality Features

Category frequencies are inspected to identify rare classes, inconsistent
labels, and possible logical issues.

In [17]:
columns_to_check = [
    "Transmission",
    "Seller Type",
    "Drivetrain",
    "Owner",
    "Fuel Type"
]

for column in columns_to_check:
    print("\n", column)
    print(df[column].value_counts(dropna=False))


 Transmission
Transmission
Manual       1133
Automatic     926
Name: count, dtype: int64

 Seller Type
Seller Type
Individual                 1997
Corporate                    57
Commercial Registration       5
Name: count, dtype: int64

 Drivetrain
Drivetrain
FWD    1330
RWD     321
AWD     272
NaN     136
Name: count, dtype: int64

 Owner
Owner
First               1619
Second               373
Third                 42
UnRegistered Car      21
Fourth                 3
4 or More              1
Name: count, dtype: int64

 Fuel Type
Fuel Type
Diesel          1049
Petrol           942
CNG               50
Electric           7
LPG                5
Hybrid             3
CNG + CNG          1
Petrol + CNG       1
Petrol + LPG       1
Name: count, dtype: int64


### Category-Frequency Observation

- Manual and Automatic transmissions both have substantial representation.
- Seller Type is highly imbalanced, with most vehicles listed by individuals.
- Drivetrain contains 136 missing values and is dominated by FWD vehicles.
- Most cars are first-owner vehicles.
- `Fourth` and `4 or More` represent similar ownership information and may
  require category standardization.
- Diesel and Petrol dominate the Fuel Type feature.
- Electric, Hybrid, LPG, and mixed-fuel categories are rare.
- `CNG + CNG` appears inconsistent and may need to be standardized as `CNG`.
- Rare categories will not be removed without evaluating their meaning and
  effect on the model.

## 15. Major Brand, Location, and Color Categories

The most frequent brands, locations, and colors are inspected to understand
dataset coverage and category imbalance.

In [18]:
print("Top 10 Makes:")
print(df["Make"].value_counts().head(10))

print("\nTop 10 Locations:")
print(df["Location"].value_counts().head(10))

print("\nColor Frequencies:")
print(df["Color"].value_counts())

Top 10 Makes:
Make
Maruti Suzuki    440
Hyundai          349
Mercedes-Benz    171
Honda            158
Toyota           132
Audi             127
BMW              121
Mahindra         119
Tata              57
Volkswagen        50
Name: count, dtype: int64

Top 10 Locations:
Location
Mumbai       342
Delhi        307
Pune         144
Bangalore    132
Hyderabad    116
Lucknow       78
Ahmedabad     70
Chennai       63
Kolkata       60
Kanpur        52
Name: count, dtype: int64

Color Frequencies:
Color
White     802
Silver    285
Grey      220
Blue      190
Black     163
Red       154
Brown      82
Maroon     37
Gold       30
Bronze     28
Green      17
Orange     16
Others     12
Yellow      9
Beige       8
Purple      5
Pink        1
Name: count, dtype: int64


### Dataset-Coverage Observation

- Maruti Suzuki and Hyundai are the most frequently represented brands.
- Mumbai and Delhi contain substantially more listings than other locations.
- White is the dominant vehicle color.
- Brand, location, and color distributions are imbalanced.
- Model conclusions will therefore reflect this dataset's listings and
  should not automatically be generalized to the complete Indian car market.
- Rare colors and locations may require grouping during preprocessing.

## 16. Working Research Questions

1. Which vehicle characteristics are most strongly associated with used-car
   listing prices?

2. Does regularization improve generalization compared with ordinary Linear
   Regression after numerical and categorical preprocessing?

3. How do Ridge, Lasso, and Elastic Net differ in coefficient shrinkage,
   feature selection, and predictive performance?

4. Does logarithmic transformation of Price improve model assumptions and
   prediction performance?

5. How sensitive are the regression results to extreme mileage values,
   high-cardinality Model information, and rare categories?

## 17. Dataset Suitability Decision

The dataset contains realistic used-car characteristics, non-uniform market
patterns, luxury-vehicle price extremes, missing technical specifications,
rare categories, and category imbalance.

Unlike the previously rejected dataset, the relationships and distributions
observed so far do not appear randomly generated. Therefore, this dataset is
accepted for the Linear and Regularized Regression portfolio project.

However, some provenance details are limited, and a small number of suspicious
mileage records require further investigation. The dataset is suitable for
methodological experimentation, portfolio development, and research training,
but it will not be used to make causal or complete Indian-market claims.

Final acceptance for modeling remains subject to the relationship analysis
and data-quality checks performed in the next EDA notebook.